# PCB Defect Detection YOLOv8 di Google Colab

Notebook ini berisi alur lengkap:
1. Mount Google Drive
2. Unzip dataset PCB
3. Audit struktur dataset
4. Perbaikan nama label yang tidak cocok
5. Validasi label YOLO
6. Training YOLOv8
7. Evaluasi dan prediksi

Aktifkan GPU dulu: **Runtime → Change runtime type → T4 GPU**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Set path dataset ZIP

Ubah `ZIP_PATH` sesuai lokasi file `archive(1).zip` di Google Drive.

In [ ]:
import os, glob, shutil, random, yaml, textwrap
from pathlib import Path

ZIP_PATH = "/content/drive/MyDrive/PCB_Datasets/archive(1).zip"
EXTRACT_DIR = "/content/pcb_dataset"
BASE = "/content/pcb_dataset/pcb-defect-dataset"

print("ZIP exists:", os.path.exists(ZIP_PATH))
print("ZIP path:", ZIP_PATH)

ZIP exists: True
ZIP path: /content/drive/MyDrive/PCB_Datasets/archive(1).zip


## 2. Extract dataset

In [ ]:
!rm -rf /content/pcb_dataset
!mkdir -p /content/pcb_dataset
!unzip -q "$ZIP_PATH" -d /content/pcb_dataset

!find /content/pcb_dataset -maxdepth 3 -type d | sort

/content/pcb_dataset
/content/pcb_dataset/pcb-defect-dataset
/content/pcb_dataset/pcb-defect-dataset/test
/content/pcb_dataset/pcb-defect-dataset/test/images
/content/pcb_dataset/pcb-defect-dataset/test/labels
/content/pcb_dataset/pcb-defect-dataset/train
/content/pcb_dataset/pcb-defect-dataset/train/images
/content/pcb_dataset/pcb-defect-dataset/train/labels
/content/pcb_dataset/pcb-defect-dataset/val
/content/pcb_dataset/pcb-defect-dataset/val/images
/content/pcb_dataset/pcb-defect-dataset/val/labels


## 3. Audit awal dataset

In [ ]:
import os, glob

def count_files():
    for split in ["train", "val", "test"]:
        img_count = len(glob.glob(f"{BASE}/{split}/images/*"))
        lbl_count = len(glob.glob(f"{BASE}/{split}/labels/*.txt"))
        print(f"{split}: images={img_count}, labels={lbl_count}")

count_files()

train: images=8534, labels=8534
val: images=1066, labels=1066
test: images=1068, labels=1068


## 4. Perbaiki mismatch nama label

Beberapa gambar bernama `light_...jpg`, sedangkan labelnya kadang bernama `l_light_...txt`. Cell ini akan menyalin label alternatif agar nama label sama dengan nama gambar.

In [ ]:
import os, glob, shutil

def repair_label_names(base):
    report = {}
    for split in ["train", "val", "test"]:
        img_dir = f"{base}/{split}/images"
        lbl_dir = f"{base}/{split}/labels"
        fixed = 0
        missing = 0
        already_ok = 0

        for img_path in glob.glob(f"{img_dir}/*"):
            if not img_path.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
                continue

            img_name = os.path.splitext(os.path.basename(img_path))[0]
            expected_label = f"{lbl_dir}/{img_name}.txt"

            if os.path.exists(expected_label):
                already_ok += 1
                continue

            candidates = [
                f"{lbl_dir}/l_{img_name}.txt",
                f"{lbl_dir}/{img_name.replace('light_', 'l_light_', 1)}.txt",
                f"{lbl_dir}/{img_name.replace('l_light_', 'light_', 1)}.txt",
            ]

            found = None
            for c in candidates:
                if os.path.exists(c):
                    found = c
                    break

            if found:
                shutil.copy(found, expected_label)
                fixed += 1
            else:
                missing += 1

        report[split] = {"already_ok": already_ok, "fixed": fixed, "missing": missing}
    return report

repair_report = repair_label_names(BASE)
print(repair_report)
count_files()

{'train': {'already_ok': 6370, 'fixed': 1706, 'missing': 458}, 'val': {'already_ok': 802, 'fixed': 26, 'missing': 238}, 'test': {'already_ok': 829, 'fixed': 25, 'missing': 214}}
train: images=8534, labels=10240
val: images=1066, labels=1092
test: images=1068, labels=1093


## 5. Hapus cache YOLO lama

Penting, karena cache lama bisa membuat YOLO tetap membaca dataset yang salah.

In [ ]:
!find /content/pcb_dataset/pcb-defect-dataset -name "*.cache" -delete
print("Cache deleted")

Cache deleted


## 6. Validasi pasangan gambar-label dan format YOLO

In [ ]:
from collections import Counter

CLASS_NAMES = [
    "mouse_bite",
    "spur",
    "missing_hole",
    "short",
    "open_circuit",
    "spurious_copper"
]

def validate_dataset(base):
    all_ok = True
    for split in ["train", "val", "test"]:
        img_dir = f"{base}/{split}/images"
        lbl_dir = f"{base}/{split}/labels"
        images = [p for p in glob.glob(f"{img_dir}/*") if p.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))]
        class_counter = Counter()
        bad_labels = []
        missing_labels = []
        empty_labels = 0
        total_boxes = 0

        for img in images:
            name = os.path.splitext(os.path.basename(img))[0]
            label = f"{lbl_dir}/{name}.txt"
            if not os.path.exists(label):
                missing_labels.append(label)
                continue
            lines = [x.strip() for x in open(label).read().splitlines() if x.strip()]
            if not lines:
                empty_labels += 1
                continue
            for line in lines:
                parts = line.split()
                if len(parts) != 5:
                    bad_labels.append((label, line, "jumlah kolom bukan 5"))
                    continue
                try:
                    cls = int(float(parts[0]))
                    vals = list(map(float, parts[1:]))
                except Exception:
                    bad_labels.append((label, line, "format angka salah"))
                    continue
                if cls < 0 or cls >= len(CLASS_NAMES):
                    bad_labels.append((label, line, "class id di luar range"))
                    continue
                if not all(0 <= v <= 1 for v in vals):
                    bad_labels.append((label, line, "koordinat bukan 0-1"))
                    continue
                class_counter[cls] += 1
                total_boxes += 1

        print("\n", split.upper())
        print("images:", len(images))
        print("missing labels:", len(missing_labels))
        print("empty labels:", empty_labels)
        print("bad label lines:", len(bad_labels))
        print("total boxes:", total_boxes)
        print("class distribution:")
        for i, name in enumerate(CLASS_NAMES):
            print(f"  {i} {name}: {class_counter[i]}")

        if missing_labels[:5]: print("contoh missing:", missing_labels[:5])
        if bad_labels[:5]: print("contoh bad:", bad_labels[:5])

        if missing_labels or bad_labels or total_boxes == 0:
            all_ok = False
    return all_ok

is_valid = validate_dataset(BASE)
print("\nDATASET VALID UNTUK TRAINING:", is_valid)


 TRAIN
images: 8534
missing labels: 458
empty labels: 0
bad label lines: 0
total boxes: 16404
class distribution:
  0 mouse_bite: 2830
  1 spur: 2754
  2 missing_hole: 2757
  3 short: 2551
  4 open_circuit: 2716
  5 spurious_copper: 2796
contoh missing: ['/content/pcb_dataset/pcb-defect-dataset/train/labels/light_07_spur_07_4_600.txt', '/content/pcb_dataset/pcb-defect-dataset/train/labels/light_07_spur_08_3_600.txt', '/content/pcb_dataset/pcb-defect-dataset/train/labels/light_06_mouse_bite_05_3_600.txt', '/content/pcb_dataset/pcb-defect-dataset/train/labels/light_04_mouse_bite_13_1_600.txt', '/content/pcb_dataset/pcb-defect-dataset/train/labels/light_09_short_05_5_600.txt']

 VAL
images: 1066
missing labels: 238
empty labels: 0
bad label lines: 0
total boxes: 1646
class distribution:
  0 mouse_bite: 292
  1 spur: 278
  2 missing_hole: 235
  3 short: 334
  4 open_circuit: 263
  5 spurious_copper: 244
contoh missing: ['/content/pcb_dataset/pcb-defect-dataset/val/labels/light_11_spurious

## 7. Buat `data.yaml` baru

In [ ]:
yaml_text = f"""path: {BASE}
train: train/images
val: val/images
test: test/images

nc: 6
names:
  - mouse_bite
  - spur
  - missing_hole
  - short
  - open_circuit
  - spurious_copper
"""

with open(f"{BASE}/data.yaml", "w") as f:
    f.write(yaml_text)

print(open(f"{BASE}/data.yaml").read())

path: /content/pcb_dataset/pcb-defect-dataset
train: train/images
val: val/images
test: test/images

nc: 6
names:
  - mouse_bite
  - spur
  - missing_hole
  - short
  - open_circuit
  - spurious_copper



## 8. Visualisasi sample bounding box

In [ ]:
import cv2, random
import matplotlib.pyplot as plt

def show_samples(split="train", n=6):
    images = glob.glob(f"{BASE}/{split}/images/*")
    images = [x for x in images if x.lower().endswith((".jpg", ".jpeg", ".png"))]
    samples = random.sample(images, min(n, len(images)))

    for img_path in samples:
        img = cv2.imread(img_path)
        if img is None:
            print("Gagal baca:", img_path)
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        name = os.path.splitext(os.path.basename(img_path))[0]
        label_path = f"{BASE}/{split}/labels/{name}.txt"

        if os.path.exists(label_path):
            for line in open(label_path):
                if not line.strip():
                    continue
                cls, x, y, bw, bh = map(float, line.split())
                x1 = int((x - bw/2) * w)
                y1 = int((y - bh/2) * h)
                x2 = int((x + bw/2) * w)
                y2 = int((y + bh/2) * h)
                cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
                cv2.putText(img, CLASS_NAMES[int(cls)], (x1, max(15, y1-5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,0,0), 1)

        plt.figure(figsize=(6,6))
        plt.imshow(img)
        plt.title(os.path.basename(img_path))
        plt.axis("off")
        plt.show()

show_samples("train", 6)

## 9. Install YOLOv8 / Ultralytics

In [ ]:
!pip install -q ultralytics
from ultralytics import YOLO
import ultralytics
ultralytics.checks()

Ultralytics 8.4.62 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 49.4/112.6 GB disk)


## 10. Training YOLOv8n

Untuk percobaan pertama gunakan 50 epoch. Kalau sudah bagus, bisa naik ke 100 epoch.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data=f"{BASE}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    patience=20,
    name="pcb_yolov8n_repaired"
)

Ultralytics 8.4.62 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/pcb_dataset/pcb-defect-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=pcb_yolov8n_repaired, nbs=64, nms=False, opset=None, optimize=False, optimizer=aut

## 11. Evaluasi pada test set

In [ ]:
best_model_path = "/content/runs/detect/pcb_yolov8n_repaired/weights/best.pt"
model = YOLO(best_model_path)

metrics = model.val(
    data=f"{BASE}/data.yaml",
    split="test"
)

Ultralytics 8.4.62 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 95.2±143.5 MB/s, size: 104.5 KB)
val: Scanning /content/pcb_dataset/pcb-defect-dataset/test/labels... 854 images, 214 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1068/1068 519.5it/s 2.1s
val: New cache created: /content/pcb_dataset/pcb-defect-dataset/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 67/67 3.6it/s 18.7s
                   all       1068       1717      0.778      0.986      0.868       0.51
            mouse_bite        135        270      0.787      0.993      0.866      0.505
                  spur        143        290      0.817      0.986      0.905      0.529
          missing_hole        148        292      0.758      0.996       0.87      0.536
                 short        146   

## 12. Prediksi gambar test

In [ ]:
model = YOLO("/content/runs/detect/pcb_yolov8n_repaired/weights/best.pt")

pred_results = model.predict(
    source=f"{BASE}/test/images",
    save=True,
    conf=0.25,
    max_det=20
)

print("Hasil prediksi tersimpan di folder runs/detect/predict")


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/1068 /content/pcb_dataset/pcb-defect-dataset/test/images/l_light_01_missing_hole_04_2_600.jpg: 640x640 2 missing_holes, 8.1ms
image 2/1068 /content/pcb_dataset/pcb-defect-dataset/test/images/l_light_01_missing_hole_07_2_600.jpg: 640x640 2 missing_holes, 7.2ms
image 3/1068 /content/pcb_dataset/pcb-defect-dataset/test/images/l_light_01_missing_hole_08_1_600.jpg: 640x640 1 missing_hole, 10.3ms
image 4/1068 /content/pcb_dataset/pcb-defect-dataset/tes

## 13. Tampilkan beberapa hasil prediksi

In [ ]:
import glob, os, cv2
import matplotlib.pyplot as plt

pred_dirs = sorted(glob.glob("/content/runs/detect/predict*"), key=os.path.getmtime)
latest_pred = pred_dirs[-1]
print("Latest prediction folder:", latest_pred)

pred_imgs = glob.glob(f"{latest_pred}/*.jpg")[:10]
for p in pred_imgs:
    img = cv2.imread(p)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(6,6))
    plt.imshow(img)
    plt.title(os.path.basename(p))
    plt.axis("off")
    plt.show()

## 14. Zip hasil training untuk download

In [ ]:
!zip -r /content/pcb_yolov8n_repaired_results.zip /content/runs/detect/pcb_yolov8n_repaired

from google.colab import files
files.download("/content/pcb_yolov8n_repaired_results.zip")

  adding: content/runs/detect/pcb_yolov8n_repaired/ (stored 0%)
  adding: content/runs/detect/pcb_yolov8n_repaired/val_batch0_labels.jpg (deflated 11%)
  adding: content/runs/detect/pcb_yolov8n_repaired/confusion_matrix.png (deflated 22%)
  adding: content/runs/detect/pcb_yolov8n_repaired/labels.jpg (deflated 26%)
  adding: content/runs/detect/pcb_yolov8n_repaired/val_batch1_labels.jpg (deflated 10%)
  adding: content/runs/detect/pcb_yolov8n_repaired/BoxR_curve.png (deflated 10%)
  adding: content/runs/detect/pcb_yolov8n_repaired/args.yaml (deflated 53%)
  adding: content/runs/detect/pcb_yolov8n_repaired/BoxF1_curve.png (deflated 10%)
  adding: content/runs/detect/pcb_yolov8n_repaired/BoxP_curve.png (deflated 9%)
  adding: content/runs/detect/pcb_yolov8n_repaired/val_batch2_pred.jpg (deflated 11%)
  adding: content/runs/detect/pcb_yolov8n_repaired/train_batch21362.jpg (deflated 6%)
  adding: content/runs/detect/pcb_yolov8n_repaired/train_batch1.jpg (deflated 3%)
  adding: content/runs/

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>